# Digital Ageing Atlas → Knowledge Graph (KG) Builder

**Source:** [Digital Ageing Atlas](http://ageing-map.org/) database  
**Species covered:** *Homo sapiens*, *Mus musculus*

## What this notebook produces

### Module 1 — Gene → Tissue edges (`Gene_Tissue`)
Maps DAA genes to BTO tissue ontology IDs.  
Gene symbols are normalised via NCBI synonym expansion before NCBI validation.

### Module 2 — Gene → BiologicalProcess edges 


---
## 0 · Configuration — edit ONLY these two lines

In [1]:
import os
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : root folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/"

# ── Derived input paths (do not edit below this line) ────────────────────────
# BTO tissue ontology: ID ↔ name
BTO_PATH         = os.path.join(BASE_PATH, "databases_for_mapping/bto/BTO_ALL_COMBINED.csv")
# UBERON anatomy ontology: ID ↔ name (available as fallback, not used by default)
UBERON_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/uberon/Uberon_ID_Name_non_dup.csv")
# ENSEMBL ↔ NCBI gene symbol crosswalk
ENS2NCBI_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
# NCBI gene_info — Human
NCBI_HUMAN_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
# NCBI gene_info — Mouse
NCBI_MOUSE_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Mus_musculus.gene_info")
# Digital Ageing Atlas main data file
DAA_PATH         = os.path.join(BASE_PATH, "digitalagingatlas/digital_ageing_atlas_data.txt")

# ── Derived output directories ───────────────────────────────────────────────
OUT_HUMAN = os.path.join(OUT_PATH, "digitalagingatlas/Human/")
OUT_MOUSE = os.path.join(OUT_PATH, "digitalagingatlas/Mouse/")

for d in [OUT_HUMAN, OUT_MOUSE]:
    os.makedirs(d, exist_ok=True)

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output root : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output root : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/


---
## 1 · Load tissue/anatomy ontology dictionaries

BTO is the primary tissue ID source. UBERON is loaded as a potential fallback.

In [2]:
# ── 1a. BTO (BRENDA Tissue Ontology) — tissue name → BTO ID ──────────────────
# Used to map tissue names from DAA to canonical BTO identifiers
BTO_file = pd.read_csv(BTO_PATH)
BTO_file_dict = dict(zip(BTO_file['name'], BTO_file['ID']))  # {tissue_name: BTO:XXXXXXX}

print(f"Loaded {len(BTO_file_dict):,} BTO tissue entries")

Loaded 6,608 BTO tissue entries


In [3]:
# ── 1b. UBERON (anatomy ontology) — name → UBERON ID ─────────────────────────
# Available as a fallback if BTO lookup fails; currently not applied in the pipeline
UBERON_file = pd.read_csv(UBERON_PATH)
UBERON_file_dict = dict(zip(UBERON_file['Name'], UBERON_file['UBERON ID']))  # {name: UBERON:XXXXXXX}

print(f"Loaded {len(UBERON_file_dict):,} UBERON anatomy entries")

Loaded 15,654 UBERON anatomy entries


---
## 2 · Build gene reference dictionaries (Human + Mouse)

Four lookups are built per species — symbol validation, synonym expansion, and description annotation.

In [4]:
# ── 2a. ENSEMBL → NCBI symbol crosswalk ──────────────────────────────────────
# Used to attach Ensembl IDs to the NCBI gene table; rows with no name are dropped.
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))  # {symbol: ENS_ID}

print(f"Loaded {len(NCBI_2ENS__dict):,} symbol → Ensembl ID mappings")
del ENS_2NCBI  # free memory — large file no longer needed

Loaded 40,940 symbol → Ensembl ID mappings


In [5]:
# ── 2b. NCBI gene_info — Human ────────────────────────────────────────────────
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)

# Symbol → description (primary lookup for validating gene symbols)
NCBI_Synbol_GENEname_dict  = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
# GeneID → description
NCBI_ALL_GENEname_dict     = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
# GeneID → symbol
NCBI_ALL_GENEIDname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))

print(f"Human NCBI: {len(NCBI_Synbol_GENEname_dict):,} symbol → description mappings")
NCBI_ALL_GENE.head(3)

Human NCBI: 193,352 symbol → description mappings


,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type,ENSEMBLE_ID
0,9606,1,A1BG,-,A1B|ABG|GAB|HYST2477,MIM:138670|HGNC:HGNC:5|Ensembl:ENSG00000121410...,19,19q13.43,alpha-1-B glycoprotein,protein-coding,A1BG,alpha-1-B glycoprotein,O,alpha-1B-glycoprotein|HEL-S-163pA|epididymis s...,20250208,-,ENSG00000121410
1,9606,2,A2M,-,A2MD|CPAMD5|FWP007|S863-7,MIM:103950|HGNC:HGNC:7|Ensembl:ENSG00000175899...,12,12p13.31,alpha-2-macroglobulin,protein-coding,A2M,alpha-2-macroglobulin,O,alpha-2-macroglobulin|C3 and PZP-like alpha-2-...,20250208,-,ENSG00000175899
2,9606,3,A2MP1,-,A2MP,HGNC:HGNC:8|Ensembl:ENSG00000291190|AllianceGe...,12,12p13.31,alpha-2-macroglobulin pseudogene 1,pseudo,A2MP1,alpha-2-macroglobulin pseudogene 1,O,pregnancy-zone protein pseudogene,20241210,-,ENSG00000291190


In [6]:
# ── 2c. Expand Human gene synonyms into a flat lookup ─────────────────────────
# The Synonyms column is pipe-separated (e.g. 'SIR2|SIRT1|hSIR2').
# We explode these so any alias resolves to the canonical NCBI Symbol.
NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))

exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for aliases, canonical in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in str(aliases).split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = canonical

print(f"Human synonym dict: {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} aliases")

Human synonym dict: 69,564 aliases


In [7]:
# ── 2d. NCBI gene_info — Mouse ────────────────────────────────────────────────
NCBI_ALL_GENE_Mouse = pd.read_csv(NCBI_MOUSE_PATH, sep='\t')

# Symbol → description
Mouse_NCBI_Synbol_GENEname_dict = dict(zip(NCBI_ALL_GENE_Mouse['Symbol'], NCBI_ALL_GENE_Mouse['description']))

print(f"Mouse NCBI: {len(Mouse_NCBI_Synbol_GENEname_dict):,} symbol → description mappings")
NCBI_ALL_GENE_Mouse.head(3)

Mouse NCBI: 112,035 symbol → description mappings


,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type
0,10090,11287,Pzp,-,A1m|A2m|MAM,MGI:MGI:87854|Ensembl:ENSMUSG00000030359|Allia...,6,6 F3|6 63.02 cM,"PZP, alpha-2-macroglobulin like",protein-coding,Pzp,"PZP, alpha-2-macroglobulin like",O,pregnancy zone protein|alpha 1 macroglobulin|a...,20250308,-
1,10090,11298,Aanat,-,AA-NAT|Nat-2|Nat4|Snat,MGI:MGI:1328365|Ensembl:ENSMUSG00000020804|All...,11,11 81.43 cM|11 E2,arylalkylamine N-acetyltransferase,protein-coding,Aanat,arylalkylamine N-acetyltransferase,O,serotonin N-acetyltransferase|aralkylamine N-a...,20250308,-
2,10090,11302,Aatk,-,AATYK|aatyk1|mKIAA0641,MGI:MGI:1197518|Ensembl:ENSMUSG00000025375|All...,11,11 E2|11 83.96 cM,apoptosis-associated tyrosine kinase,protein-coding,Aatk,apoptosis-associated tyrosine kinase,O,serine/threonine-protein kinase LMTK1|apoptosi...,20250308,-


In [8]:
# ── 2e. Expand Mouse gene synonyms into a flat lookup ─────────────────────────
Mouse_NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE_Mouse['Synonyms'], NCBI_ALL_GENE_Mouse['Symbol']))

exploded_dict_Mouse_NCBI_ALL_GENE_Syn_Dict = {}
for aliases, canonical in Mouse_NCBI_ALL_GENE_Syn_Dict.items():
    for alias in str(aliases).split('|'):
        exploded_dict_Mouse_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = canonical

print(f"Mouse synonym dict: {len(exploded_dict_Mouse_NCBI_ALL_GENE_Syn_Dict):,} aliases")

Mouse synonym dict: 73,486 aliases


---
## 3 · Load and parse Digital Ageing Atlas data

The source file stores gene entries as `SYMBOL (Full Name)`, e.g. `SLC2A1 (Glucose transporter 1)`.  
We split this into two columns: `Gene` (symbol) and `Gene_name` (full name).

This parsed table is reused by both Module 1 (Gene→Tissue) and Module 2 (Gene→BiologicalProcess).

In [9]:
# ── 3. Load and parse DAA source file ─────────────────────────────────────────
Human_gene_data = pd.read_csv(DAA_PATH, sep='\t')

# Drop rows with no species annotation
Human_gene_data = Human_gene_data[~Human_gene_data['Species'].isna()]

# Gene column format: 'SYMBOL (Full gene name)'
# Split into symbol and full name using ' (' as the delimiter
Human_gene_data['Gene_name'] = Human_gene_data['Gene'].str.split(' \(', expand=True)[1]
Human_gene_data['Gene']      = Human_gene_data['Gene'].str.split(' \(', expand=True)[0]
# Remove the trailing ')' from the gene name
Human_gene_data['Gene_name'] = Human_gene_data['Gene_name'].str.split('\)', expand=True)[0]

# Drop any rows where gene symbol could not be parsed
Human_gene_data = Human_gene_data[~Human_gene_data['Gene'].isna()]

print(f"Loaded {len(Human_gene_data):,} rows")
print("Species present:", set(Human_gene_data['Species']))
print("Columns:", list(Human_gene_data.columns))
Human_gene_data

Loaded 3,699 rows
Species present: {'Mus musculus', 'Homo sapiens'}
Columns: ['Identifier', 'Change name', 'Change type', 'Species', 'Change gender', 'Age change starts', 'Age change ends', 'Description', 'Tissues', 'Gene', 'Properties', 'Type of data', 'Process measured', 'Sample size', 'Method of collection', 'Data transforms', 'Percentage change', 'P value', 'Coefficiant', 'Intercept', 'Relationship parent identifiers', 'References (with LibAge reference ID in brackets)', 'Gene_name']


,Identifier,Change name,Change type,Species,Change gender,Age change starts,Age change ends,Description,Tissues,Gene,...,Sample size,Method of collection,Data transforms,Percentage change,P value,Coefficiant,Intercept,Relationship parent identifiers,References (with LibAge reference ID in brackets),Gene_name
0,DAA1359,4-aminobutyrate aminotransferase,molecular,Mus musculus,male/female,3.0,23.0,NaN,Hematological System,Abat,...,20,Microarray,Log2,45.0,0.000277,NaN,NaN,NaN,"2882: Rossi et al. (2005) ""Cell intrinsic alte...",4-aminobutyrate aminotransferase
1,DAA318,5'-3' exoribonuclease 1,molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,ERI1,...,0,Microarray,Log,4.0,0.000131,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...",exoribonuclease 1
3,DAA350,"5'-nucleotidase, cytosolic II",molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,NT5C2,...,0,Microarray,Log,5.0,0.000182,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...","5'-nucleotidase, cytosolic II"
4,DAA335,6-pyruvoyltetrahydropterin synthase,molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,PTS,...,0,Microarray,Log,-3.0,0.000236,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...",6-pyruvoyltetrahydropterin synthase
5,DAA1512,7-dehydrocholesterol reductase,molecular,Mus musculus,male/female,6.0,22.0,NaN,Liver,Dhcr7,...,20,Microarray,Log2,19.0,0.000131,NaN,NaN,NaN,"1334: Papaconstantinou et al. (2005) ""Hepatic ...",7-dehydrocholesterol reductase
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4245,DAA1615,zinc finger protein 84,molecular,Mus musculus,male/female,2.0,26.0,NaN,Lung,Zfp84,...,20,Microarray,Log2,-2.0,0.000541,NaN,NaN,NaN,"2785: Misra et al. (2007) ""Global expression p...",zinc finger protein 84
4246,DAA2102,zinc finger RNA binding protein increases with...,molecular,Homo sapiens,female,-1.0,-1.0,Gene expression showed a significant (p<E-3) i...,Ovary,ZFR,...,NaN,NaN,NaN,150.0,0.005680,NaN,NaN,NaN,"2714: Grøndahl et al. (2010) ""Gene expression ...",zinc finger RNA binding protein
4247,DAA253,zinc fingers and homeoboxes 2,molecular,Homo sapiens,male/female,26.0,106.0,NaN,Brain,ZHX2,...,0,Microarray,Log,20.0,0.000108,NaN,NaN,NaN,"2742: Lu et al. (2004) ""Gene regulation and DN...",zinc fingers and homeoboxes 2
4248,DAA1891,"zinc finger, ZZ domain containing 3",molecular,Mus musculus,male/female,5.0,25.0,NaN,Skeletal Muscle,Zzz3,...,20,Microarray,Log2,-12.0,0.000420,NaN,NaN,NaN,"2648: NCBI GEO Dataset (2006) ""Caloric restric...","zinc finger, ZZ domain containing 3"


In [10]:
Human_gene_data[Human_gene_data['Percentage change'] == 3385.0]

,Identifier,Change name,Change type,Species,Change gender,Age change starts,Age change ends,Description,Tissues,Gene,...,Sample size,Method of collection,Data transforms,Percentage change,P value,Coefficiant,Intercept,Relationship parent identifiers,References (with LibAge reference ID in brackets),Gene_name
732,DAA1253,"eosinophil-associated, ribonuclease A family, ...",molecular,Mus musculus,male/female,5.0,22.0,NaN,Brain,Ear5,...,20,Microarray,Log2,3385.0,0.000739,NaN,NaN,NaN,"2765: Godbout et al. (2005) ""Exaggerated neuro...","eosinophil-associated, ribonuclease A family, ..."


In [11]:
Human_gene_data['Data transforms'].value_counts()

Data transforms
Log2    627
Log     586
Name: count, dtype: int64

In [12]:
max(Human_gene_data['Percentage change']), min(Human_gene_data['Percentage change'])

(3385.0, -98.7)

---
## 4 · Module 1 — Gene → Tissue edges

**Pipeline:**  
`Extract Gene + Tissue columns` → `Lowercase & normalise tissue names` → `Map tissue → BTO ID` →  
`Drop unresolved tissues` → `For each species: normalise gene symbols via synonyms → validate vs NCBI → add schema → export`

In [13]:
# ── 4a. Build Gene–Tissue base table ─────────────────────────────────────────
DAA_GENE_TISSUE = Human_gene_data[['Gene', 'Tissues', 'Species']].copy()
DAA_GENE_TISSUE['Relation'] = 'Gene_Tissue'
DAA_GENE_TISSUE.rename(columns={'Gene': 'Head', 'Tissues': 'Tail_detail_name'}, inplace=True)

# Lowercase tissue names for consistent BTO lookup
DAA_GENE_TISSUE['Tail_detail_name'] = DAA_GENE_TISSUE['Tail_detail_name'].str.lower()

# Normalise DAA tissue name variants to match BTO vocabulary
DAA_GENE_TISSUE['Tail_detail_name'] = DAA_GENE_TISSUE['Tail_detail_name'].replace({
    'hematological system': 'hematopoietic system',  # DAA variant → BTO canonical
    'plasma':               'blood plasma'            # DAA variant → BTO canonical
})

# Map tissue name → BTO ID
DAA_GENE_TISSUE['Tail'] = DAA_GENE_TISSUE['Tail_detail_name'].map(BTO_file_dict)

# Drop rows where no BTO ID could be resolved
before = len(DAA_GENE_TISSUE)
DAA_GENE_TISSUE = DAA_GENE_TISSUE[~DAA_GENE_TISSUE['Tail'].isna()]
print(f"BTO tissue resolved: {len(DAA_GENE_TISSUE):,} kept / {before - len(DAA_GENE_TISSUE):,} dropped")
print("Species distribution:")
print(DAA_GENE_TISSUE['Species'].value_counts())

BTO tissue resolved: 3,513 kept / 186 dropped
Species distribution:
Species
Homo sapiens    2928
Mus musculus     585
Name: count, dtype: int64


### 4b · Human — Gene → Tissue

In [14]:
# ── Filter to Human rows ──────────────────────────────────────────────────────
DAA_GENE_TISSUE_Human = DAA_GENE_TISSUE[DAA_GENE_TISSUE['Species'] == 'Homo sapiens'].copy()

# Normalise gene symbols via synonym expansion:
# Some DAA entries use non-canonical aliases; map them to the NCBI canonical symbol first.
# fillna keeps the original symbol if no synonym match is found.
DAA_GENE_TISSUE_Human['Head_mapped'] = (
    DAA_GENE_TISSUE_Human['Head']
    .map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
    .fillna(DAA_GENE_TISSUE_Human['Head'])
)
# Swap: Head_mapped (canonical) → Head, old Head → Head_mapped (alias, kept for audit)
DAA_GENE_TISSUE_Human[['Head', 'Head_mapped']] = DAA_GENE_TISSUE_Human[['Head_mapped', 'Head']]

# Validate against NCBI: map symbol → description; drop any with no NCBI entry
DAA_GENE_TISSUE_Human['Head_detail_name'] = DAA_GENE_TISSUE_Human['Head'].map(NCBI_Synbol_GENEname_dict)
before = len(DAA_GENE_TISSUE_Human)
DAA_GENE_TISSUE_Human = DAA_GENE_TISSUE_Human[~DAA_GENE_TISSUE_Human['Head_detail_name'].isna()]
print(f"Human NCBI validation: {len(DAA_GENE_TISSUE_Human):,} kept / {before - len(DAA_GENE_TISSUE_Human):,} dropped")

Human NCBI validation: 2,917 kept / 11 dropped


In [15]:
# ── Add KG schema metadata ────────────────────────────────────────────────────
DAA_GENE_TISSUE_Human['Head_type']       = 'Gene'
DAA_GENE_TISSUE_Human['Tail_type']       = 'Tissue'
DAA_GENE_TISSUE_Human['Head_ID_IS']      = 'NCBI_ID'
DAA_GENE_TISSUE_Human['Tail_ID_IS']      = 'BTO_ID'
DAA_GENE_TISSUE_Human['Species']         = 'Homo sapiens'
DAA_GENE_TISSUE_Human['Relation_Source'] = 'DigitalAgingAtlas'

# Select only columns that exist in this DataFrame (safe for schema evolution)
desired_cols = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Relation_type', 'Tail_type',
    'Relation_Source', 'KG_Source', 'Species',
    'Head_Alt_ID', 'Head_detail_name', 'Head_SMILES_name',
    'Tail_DO_Alt_ids', 'Tail_IUPAC', 'Tail_Smiles', 'Tail_detail_name',
    'Head_ID_IS', 'Tail_ID_IS', 'Kg_index', 'Kg', 'Change'
]
DAA_GENE_TISSUE_Human = DAA_GENE_TISSUE_Human[[c for c in desired_cols if c in DAA_GENE_TISSUE_Human.columns]]

print(f"Final Human Gene–Tissue shape: {DAA_GENE_TISSUE_Human.shape}")
DAA_GENE_TISSUE_Human.head(3)

Final Human Gene–Tissue shape: (2917, 11)


,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
1,ERI1,Gene_Tissue,BTO:0001103,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,exoribonuclease 1,skeletal muscle,NCBI_ID,BTO_ID
3,NT5C2,Gene_Tissue,BTO:0001103,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,"5'-nucleotidase, cytosolic II",skeletal muscle,NCBI_ID,BTO_ID
4,PTS,Gene_Tissue,BTO:0001103,Gene,Tissue,DigitalAgingAtlas,Homo sapiens,6-pyruvoyltetrahydropterin synthase,skeletal muscle,NCBI_ID,BTO_ID


In [16]:
# ── Export ────────────────────────────────────────────────────────────────────
out_path = os.path.join(OUT_HUMAN, 'DAA_Gene_Tissue_Human.csv')
DAA_GENE_TISSUE_Human.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

Saved → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/digitalagingatlas/Human/DAA_Gene_Tissue_Human.csv


### 4c · Mouse — Gene → Tissue

In [17]:
# ── Filter to Mouse rows ──────────────────────────────────────────────────────
DAA_GENE_TISSUE_Mouse = DAA_GENE_TISSUE[DAA_GENE_TISSUE['Species'] == 'Mus musculus'].copy()

# Normalise gene symbols via Mouse-specific synonym expansion
DAA_GENE_TISSUE_Mouse['Head_mapped'] = (
    DAA_GENE_TISSUE_Mouse['Head']
    .map(exploded_dict_Mouse_NCBI_ALL_GENE_Syn_Dict)
    .fillna(DAA_GENE_TISSUE_Mouse['Head'])
)
DAA_GENE_TISSUE_Mouse[['Head', 'Head_mapped']] = DAA_GENE_TISSUE_Mouse[['Head_mapped', 'Head']]

# Validate against Mouse NCBI
DAA_GENE_TISSUE_Mouse['Head_detail_name'] = DAA_GENE_TISSUE_Mouse['Head'].map(Mouse_NCBI_Synbol_GENEname_dict)
before = len(DAA_GENE_TISSUE_Mouse)
DAA_GENE_TISSUE_Mouse = DAA_GENE_TISSUE_Mouse[~DAA_GENE_TISSUE_Mouse['Head_detail_name'].isna()]
print(f"Mouse NCBI validation: {len(DAA_GENE_TISSUE_Mouse):,} kept / {before - len(DAA_GENE_TISSUE_Mouse):,} dropped")

Mouse NCBI validation: 576 kept / 9 dropped


In [18]:
# ── Add KG schema metadata ────────────────────────────────────────────────────
DAA_GENE_TISSUE_Mouse['Head_type']       = 'Gene'
DAA_GENE_TISSUE_Mouse['Tail_type']       = 'Tissue'
DAA_GENE_TISSUE_Mouse['Head_ID_IS']      = 'NCBI_ID'
DAA_GENE_TISSUE_Mouse['Tail_ID_IS']      = 'BTO_ID'
DAA_GENE_TISSUE_Mouse['Species']         = 'Mus musculus'
DAA_GENE_TISSUE_Mouse['Relation_Source'] = 'DigitalAgingAtlas'

DAA_GENE_TISSUE_Mouse = DAA_GENE_TISSUE_Mouse[[c for c in desired_cols if c in DAA_GENE_TISSUE_Mouse.columns]]

print(f"Final Mouse Gene–Tissue shape: {DAA_GENE_TISSUE_Mouse.shape}")
DAA_GENE_TISSUE_Mouse.head(3)

Final Mouse Gene–Tissue shape: (576, 11)


,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,Abat,Gene_Tissue,BTO:0000570,Gene,Tissue,DigitalAgingAtlas,Mus musculus,4-aminobutyrate aminotransferase,hematopoietic system,NCBI_ID,BTO_ID
5,Dhcr7,Gene_Tissue,BTO:0000759,Gene,Tissue,DigitalAgingAtlas,Mus musculus,7-dehydrocholesterol reductase,liver,NCBI_ID,BTO_ID
6,Abhd14a,Gene_Tissue,BTO:0000887,Gene,Tissue,DigitalAgingAtlas,Mus musculus,abhydrolase domain containing 14A,muscle,NCBI_ID,BTO_ID


In [19]:
# ── Export ────────────────────────────────────────────────────────────────────
out_path = os.path.join(OUT_MOUSE, 'DAA_Gene_Tissue_Mouse.csv')
DAA_GENE_TISSUE_Mouse.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

Saved → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/digitalagingatlas/Mouse/DAA_Gene_Tissue_Mouse.csv


---
## 5 · Module 2 — Gene → BiologicalProcess edges



In [20]:
# ── 5a. Reload and parse the DAA source file for the second module ────────────
# Re-read so this module is fully independent and can be run without Module 1.
DAA_gene_data = pd.read_csv(DAA_PATH, sep='\t')
DAA_gene_data = DAA_gene_data[~DAA_gene_data['Species'].isna()]

# Same parsing as Module 1: split 'SYMBOL (Full name)' into two columns
DAA_gene_data['Gene_name'] = DAA_gene_data['Gene'].str.split(' \(', expand=True)[1]
DAA_gene_data['Gene']      = DAA_gene_data['Gene'].str.split(' \(', expand=True)[0]
DAA_gene_data['Gene_name'] = DAA_gene_data['Gene_name'].str.split('\)', expand=True)[0]
DAA_gene_data = DAA_gene_data[~DAA_gene_data['Gene'].isna()]

print(f"Loaded {len(DAA_gene_data):,} rows")
print("Species:", DAA_gene_data['Species'].value_counts().to_dict())

# Step 1: Drop rows with NaN in 'Data transforms'
DAA_gene_data = DAA_gene_data[DAA_gene_data['Data transforms'].notna()]

# Step 2: Drop rows with NaN in 'Percentage change'
DAA_gene_data = DAA_gene_data[DAA_gene_data['Percentage change'].notna()]

DAA_gene_data


Loaded 3,699 rows
Species: {'Homo sapiens': 3064, 'Mus musculus': 635}


,Identifier,Change name,Change type,Species,Change gender,Age change starts,Age change ends,Description,Tissues,Gene,...,Sample size,Method of collection,Data transforms,Percentage change,P value,Coefficiant,Intercept,Relationship parent identifiers,References (with LibAge reference ID in brackets),Gene_name
0,DAA1359,4-aminobutyrate aminotransferase,molecular,Mus musculus,male/female,3.0,23.0,NaN,Hematological System,Abat,...,20,Microarray,Log2,45.0,0.000277,NaN,NaN,NaN,"2882: Rossi et al. (2005) ""Cell intrinsic alte...",4-aminobutyrate aminotransferase
1,DAA318,5'-3' exoribonuclease 1,molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,ERI1,...,0,Microarray,Log,4.0,0.000131,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...",exoribonuclease 1
3,DAA350,"5'-nucleotidase, cytosolic II",molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,NT5C2,...,0,Microarray,Log,5.0,0.000182,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...","5'-nucleotidase, cytosolic II"
4,DAA335,6-pyruvoyltetrahydropterin synthase,molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,PTS,...,0,Microarray,Log,-3.0,0.000236,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...",6-pyruvoyltetrahydropterin synthase
5,DAA1512,7-dehydrocholesterol reductase,molecular,Mus musculus,male/female,6.0,22.0,NaN,Liver,Dhcr7,...,20,Microarray,Log2,19.0,0.000131,NaN,NaN,NaN,"1334: Papaconstantinou et al. (2005) ""Hepatic ...",7-dehydrocholesterol reductase
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4243,DAA1276,zinc finger protein 68,molecular,Mus musculus,male/female,5.0,22.0,NaN,Brain,Zfp68,...,20,Microarray,Log2,5.0,0.000844,NaN,NaN,NaN,"2765: Godbout et al. (2005) ""Exaggerated neuro...",zinc finger protein 68
4245,DAA1615,zinc finger protein 84,molecular,Mus musculus,male/female,2.0,26.0,NaN,Lung,Zfp84,...,20,Microarray,Log2,-2.0,0.000541,NaN,NaN,NaN,"2785: Misra et al. (2007) ""Global expression p...",zinc finger protein 84
4247,DAA253,zinc fingers and homeoboxes 2,molecular,Homo sapiens,male/female,26.0,106.0,NaN,Brain,ZHX2,...,0,Microarray,Log,20.0,0.000108,NaN,NaN,NaN,"2742: Lu et al. (2004) ""Gene regulation and DN...",zinc fingers and homeoboxes 2
4248,DAA1891,"zinc finger, ZZ domain containing 3",molecular,Mus musculus,male/female,5.0,25.0,NaN,Skeletal Muscle,Zzz3,...,20,Microarray,Log2,-12.0,0.000420,NaN,NaN,NaN,"2648: NCBI GEO Dataset (2006) ""Caloric restric...","zinc finger, ZZ domain containing 3"


In [21]:
# Step 3: Assign relation using threshold > 1 / < -1
DAA_gene_data['Relation'] = DAA_gene_data['Percentage change'].apply(
    lambda x: 'PositivelyAssociatedWith'  if x > 1
    else ('NegativelyAssociatedWith'       if x < -1
    else  'NotAssociatedWith')
)
DAA_gene_data

,Identifier,Change name,Change type,Species,Change gender,Age change starts,Age change ends,Description,Tissues,Gene,...,Method of collection,Data transforms,Percentage change,P value,Coefficiant,Intercept,Relationship parent identifiers,References (with LibAge reference ID in brackets),Gene_name,Relation
0,DAA1359,4-aminobutyrate aminotransferase,molecular,Mus musculus,male/female,3.0,23.0,NaN,Hematological System,Abat,...,Microarray,Log2,45.0,0.000277,NaN,NaN,NaN,"2882: Rossi et al. (2005) ""Cell intrinsic alte...",4-aminobutyrate aminotransferase,PositivelyAssociatedWith
1,DAA318,5'-3' exoribonuclease 1,molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,ERI1,...,Microarray,Log,4.0,0.000131,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...",exoribonuclease 1,PositivelyAssociatedWith
3,DAA350,"5'-nucleotidase, cytosolic II",molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,NT5C2,...,Microarray,Log,5.0,0.000182,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...","5'-nucleotidase, cytosolic II",PositivelyAssociatedWith
4,DAA335,6-pyruvoyltetrahydropterin synthase,molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,PTS,...,Microarray,Log,-3.0,0.000236,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...",6-pyruvoyltetrahydropterin synthase,NegativelyAssociatedWith
5,DAA1512,7-dehydrocholesterol reductase,molecular,Mus musculus,male/female,6.0,22.0,NaN,Liver,Dhcr7,...,Microarray,Log2,19.0,0.000131,NaN,NaN,NaN,"1334: Papaconstantinou et al. (2005) ""Hepatic ...",7-dehydrocholesterol reductase,PositivelyAssociatedWith
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4243,DAA1276,zinc finger protein 68,molecular,Mus musculus,male/female,5.0,22.0,NaN,Brain,Zfp68,...,Microarray,Log2,5.0,0.000844,NaN,NaN,NaN,"2765: Godbout et al. (2005) ""Exaggerated neuro...",zinc finger protein 68,PositivelyAssociatedWith
4245,DAA1615,zinc finger protein 84,molecular,Mus musculus,male/female,2.0,26.0,NaN,Lung,Zfp84,...,Microarray,Log2,-2.0,0.000541,NaN,NaN,NaN,"2785: Misra et al. (2007) ""Global expression p...",zinc finger protein 84,NegativelyAssociatedWith
4247,DAA253,zinc fingers and homeoboxes 2,molecular,Homo sapiens,male/female,26.0,106.0,NaN,Brain,ZHX2,...,Microarray,Log,20.0,0.000108,NaN,NaN,NaN,"2742: Lu et al. (2004) ""Gene regulation and DN...",zinc fingers and homeoboxes 2,PositivelyAssociatedWith
4248,DAA1891,"zinc finger, ZZ domain containing 3",molecular,Mus musculus,male/female,5.0,25.0,NaN,Skeletal Muscle,Zzz3,...,Microarray,Log2,-12.0,0.000420,NaN,NaN,NaN,"2648: NCBI GEO Dataset (2006) ""Caloric restric...","zinc finger, ZZ domain containing 3",NegativelyAssociatedWith


In [22]:
# Fix GO term
DAA_gene_data['Tail']             = 'GO:0007568'
DAA_gene_data['Tail_detail_name'] = 'aging'
DAA_gene_data['Tail_type']        = 'BiologicalProcess'
DAA_gene_data['Data_type']        = 'Associative'
DAA_gene_data

,Identifier,Change name,Change type,Species,Change gender,Age change starts,Age change ends,Description,Tissues,Gene,...,Coefficiant,Intercept,Relationship parent identifiers,References (with LibAge reference ID in brackets),Gene_name,Relation,Tail,Tail_detail_name,Tail_type,Data_type
0,DAA1359,4-aminobutyrate aminotransferase,molecular,Mus musculus,male/female,3.0,23.0,NaN,Hematological System,Abat,...,NaN,NaN,NaN,"2882: Rossi et al. (2005) ""Cell intrinsic alte...",4-aminobutyrate aminotransferase,PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative
1,DAA318,5'-3' exoribonuclease 1,molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,ERI1,...,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...",exoribonuclease 1,PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative
3,DAA350,"5'-nucleotidase, cytosolic II",molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,NT5C2,...,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...","5'-nucleotidase, cytosolic II",PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative
4,DAA335,6-pyruvoyltetrahydropterin synthase,molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,PTS,...,NaN,NaN,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...",6-pyruvoyltetrahydropterin synthase,NegativelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative
5,DAA1512,7-dehydrocholesterol reductase,molecular,Mus musculus,male/female,6.0,22.0,NaN,Liver,Dhcr7,...,NaN,NaN,NaN,"1334: Papaconstantinou et al. (2005) ""Hepatic ...",7-dehydrocholesterol reductase,PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4243,DAA1276,zinc finger protein 68,molecular,Mus musculus,male/female,5.0,22.0,NaN,Brain,Zfp68,...,NaN,NaN,NaN,"2765: Godbout et al. (2005) ""Exaggerated neuro...",zinc finger protein 68,PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative
4245,DAA1615,zinc finger protein 84,molecular,Mus musculus,male/female,2.0,26.0,NaN,Lung,Zfp84,...,NaN,NaN,NaN,"2785: Misra et al. (2007) ""Global expression p...",zinc finger protein 84,NegativelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative
4247,DAA253,zinc fingers and homeoboxes 2,molecular,Homo sapiens,male/female,26.0,106.0,NaN,Brain,ZHX2,...,NaN,NaN,NaN,"2742: Lu et al. (2004) ""Gene regulation and DN...",zinc fingers and homeoboxes 2,PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative
4248,DAA1891,"zinc finger, ZZ domain containing 3",molecular,Mus musculus,male/female,5.0,25.0,NaN,Skeletal Muscle,Zzz3,...,NaN,NaN,NaN,"2648: NCBI GEO Dataset (2006) ""Caloric restric...","zinc finger, ZZ domain containing 3",NegativelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative


In [23]:
# ── 5b. Rename and build initial KG columns ───────────────────────────────────
DAA_gene_data = DAA_gene_data.rename(columns={'Gene': 'Head'})

# Keep the 'Change name' column as Head_details for human-readable context
DAA_gene_data['Head_details'] = DAA_gene_data['Change name']
DAA_gene_data['Head_type']    = 'Gene'
DAA_gene_data

,Identifier,Change name,Change type,Species,Change gender,Age change starts,Age change ends,Description,Tissues,Head,...,Relationship parent identifiers,References (with LibAge reference ID in brackets),Gene_name,Relation,Tail,Tail_detail_name,Tail_type,Data_type,Head_details,Head_type
0,DAA1359,4-aminobutyrate aminotransferase,molecular,Mus musculus,male/female,3.0,23.0,NaN,Hematological System,Abat,...,NaN,"2882: Rossi et al. (2005) ""Cell intrinsic alte...",4-aminobutyrate aminotransferase,PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative,4-aminobutyrate aminotransferase,Gene
1,DAA318,5'-3' exoribonuclease 1,molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,ERI1,...,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...",exoribonuclease 1,PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative,5'-3' exoribonuclease 1,Gene
3,DAA350,"5'-nucleotidase, cytosolic II",molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,NT5C2,...,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...","5'-nucleotidase, cytosolic II",PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative,"5'-nucleotidase, cytosolic II",Gene
4,DAA335,6-pyruvoyltetrahydropterin synthase,molecular,Homo sapiens,male/female,20.0,75.0,NaN,Skeletal Muscle,PTS,...,NaN,"2828: Welle et al. (2004) ""Skeletal muscle gen...",6-pyruvoyltetrahydropterin synthase,NegativelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative,6-pyruvoyltetrahydropterin synthase,Gene
5,DAA1512,7-dehydrocholesterol reductase,molecular,Mus musculus,male/female,6.0,22.0,NaN,Liver,Dhcr7,...,NaN,"1334: Papaconstantinou et al. (2005) ""Hepatic ...",7-dehydrocholesterol reductase,PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative,7-dehydrocholesterol reductase,Gene
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4243,DAA1276,zinc finger protein 68,molecular,Mus musculus,male/female,5.0,22.0,NaN,Brain,Zfp68,...,NaN,"2765: Godbout et al. (2005) ""Exaggerated neuro...",zinc finger protein 68,PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative,zinc finger protein 68,Gene
4245,DAA1615,zinc finger protein 84,molecular,Mus musculus,male/female,2.0,26.0,NaN,Lung,Zfp84,...,NaN,"2785: Misra et al. (2007) ""Global expression p...",zinc finger protein 84,NegativelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative,zinc finger protein 84,Gene
4247,DAA253,zinc fingers and homeoboxes 2,molecular,Homo sapiens,male/female,26.0,106.0,NaN,Brain,ZHX2,...,NaN,"2742: Lu et al. (2004) ""Gene regulation and DN...",zinc fingers and homeoboxes 2,PositivelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative,zinc fingers and homeoboxes 2,Gene
4248,DAA1891,"zinc finger, ZZ domain containing 3",molecular,Mus musculus,male/female,5.0,25.0,NaN,Skeletal Muscle,Zzz3,...,NaN,"2648: NCBI GEO Dataset (2006) ""Caloric restric...","zinc finger, ZZ domain containing 3",NegativelyAssociatedWith,GO:0007568,aging,BiologicalProcess,Associative,"zinc finger, ZZ domain containing 3",Gene


In [24]:
desired_order = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Tail_type',
    'Head_detail_name', 'Head_SMILE_name',
    'Relation_Source', 'Species', 'Head_Alt_name', 'Data_type',
    'Dosage', 'avg_lifespan_change_percent', 'pubmed_id'
]
DAA_gene_data
DAA_gene_data = DAA_gene_data[[c for c in desired_order if c in DAA_gene_data.columns]]
DAA_gene_data

,Head,Relation,Tail,Head_type,Tail_type,Species,Data_type
0,Abat,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Mus musculus,Associative
1,ERI1,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative
3,NT5C2,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative
4,PTS,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative
5,Dhcr7,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Mus musculus,Associative
...,...,...,...,...,...,...,...
4243,Zfp68,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Mus musculus,Associative
4245,Zfp84,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Mus musculus,Associative
4247,ZHX2,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative
4248,Zzz3,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Mus musculus,Associative


In [25]:
print("F2 Relation distribution:")
print(DAA_gene_data['Relation'].value_counts())
print("\nF2 Species distribution:")
print(DAA_gene_data['Species'].value_counts())

F2 Relation distribution:
Relation
NegativelyAssociatedWith    600
PositivelyAssociatedWith    586
NotAssociatedWith            27
Name: count, dtype: int64

F2 Species distribution:
Species
Mus musculus    627
Homo sapiens    586
Name: count, dtype: int64


### 5d · Human — Gene → BiologicalProcess

In [26]:
# ── Filter to Human, validate gene symbols against Human NCBI ─────────────────
Human_Gene_BiologicalProcess = DAA_gene_data[DAA_gene_data['Species'] == 'Homo sapiens'].copy()

Human_Gene_BiologicalProcess['Head_detail_name'] = Human_Gene_BiologicalProcess['Head'].map(NCBI_Synbol_GENEname_dict)
before = len(Human_Gene_BiologicalProcess)
Human_Gene_BiologicalProcess = Human_Gene_BiologicalProcess[~Human_Gene_BiologicalProcess['Head_detail_name'].isna()]
print(f"Human NCBI validation: {len(Human_Gene_BiologicalProcess):,} kept / {before - len(Human_Gene_BiologicalProcess):,} dropped")
Human_Gene_BiologicalProcess

Human NCBI validation: 526 kept / 60 dropped


,Head,Relation,Tail,Head_type,Tail_type,Species,Data_type,Head_detail_name
1,ERI1,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative,exoribonuclease 1
3,NT5C2,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative,"5'-nucleotidase, cytosolic II"
4,PTS,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative,6-pyruvoyltetrahydropterin synthase
7,ABHD14A,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative,abhydrolase domain containing 14A
9,ABI2,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative,abl interactor 2
...,...,...,...,...,...,...,...,...
4227,ZMIZ2,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative,zinc finger MIZ-type containing 2
4233,ZNF280A,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative,zinc finger protein 280A
4234,ZFP36L1,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative,ZFP36 ring finger protein like 1
4236,ZNF507,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,Homo sapiens,Associative,zinc finger protein 507


In [27]:
Human_Gene_BiologicalProcess['Relation'].value_counts(), #Human_Gene_BiologicalProcess['Relation'].value_counts() 

(Relation
 NegativelyAssociatedWith    310
 PositivelyAssociatedWith    215
 NotAssociatedWith             1
 Name: count, dtype: int64,)

In [28]:
# ── Add KG schema metadata ────────────────────────────────────────────────────
Human_Gene_BiologicalProcess['Head_type']       = 'Gene'
Human_Gene_BiologicalProcess['Tail_type']       = 'BiologicalProcess'
Human_Gene_BiologicalProcess['Head_ID_IS']      = 'NCBI_ID'
Human_Gene_BiologicalProcess['Tail_ID_IS']      = 'Quick_GO'
Human_Gene_BiologicalProcess['Species']         = 'Homo sapiens'
Human_Gene_BiologicalProcess['Relation_Source'] = 'DigitalAgingAtlas'


desired_cols = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Tail_type',
    'Head_detail_name', 'Head_SMILE_name',
    'Relation_Source', 'Species', 'Head_Alt_name', 'Data_type',
    'Dosage', 'avg_lifespan_change_percent', 'pubmed_id'
]
# Reuse the same desired_cols schema as Module 1
Human_Gene_Phenotype = Human_Gene_BiologicalProcess[
    [c for c in desired_cols if c in Human_Gene_BiologicalProcess.columns]
]

print(f"Final Human Gene–BiologicalProcess shape: {Human_Gene_Phenotype.shape}")
print(Human_Gene_Phenotype['Relation'].value_counts())

Final Human Gene–BiologicalProcess shape: (526, 9)
Relation
NegativelyAssociatedWith    310
PositivelyAssociatedWith    215
NotAssociatedWith             1
Name: count, dtype: int64


In [29]:
OUT_HUMAN

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/digitalagingatlas/Human/'

In [30]:
# ── Split by relation direction and export ────────────────────────────────────
Human_PositivelyAssociatedWith = Human_Gene_Phenotype[Human_Gene_Phenotype['Relation'] == 'PositivelyAssociatedWith']
Human_NegativelyAssociatedWith = Human_Gene_Phenotype[Human_Gene_Phenotype['Relation'] == 'NegativelyAssociatedWith']
Human_NotAssociatedWith = Human_Gene_Phenotype[Human_Gene_Phenotype['Relation'] == 'NotAssociatedWith']

Human_PositivelyAssociatedWith.to_csv(os.path.join(OUT_HUMAN, 'DAA_Gene_PositivelyAssociatedWith_BiologicalProcess_Human.csv'), index=False)
Human_NegativelyAssociatedWith.to_csv(os.path.join(OUT_HUMAN, 'DAA_Gene_NegativelyAssociatedWith_BiologicalProcess_Human.csv'), index=False)
Human_NotAssociatedWith.to_csv(os.path.join(OUT_HUMAN, 'DAA_Gene_NotAssociatedWith_BiologicalProcess_Human.csv'), index=False)

print(f"Saved 3 files → {OUT_HUMAN}")

Saved 3 files → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/digitalagingatlas/Human/


In [31]:

print(f"\nSaved 3 Human files:")
print(f"  Pos: {len(Human_PositivelyAssociatedWith):,} rows")
print(f"  Neg: {len(Human_NegativelyAssociatedWith):,} rows")
print(f"  Not: {len(Human_NotAssociatedWith):,} rows")


Saved 3 Human files:
  Pos: 215 rows
  Neg: 310 rows
  Not: 1 rows


In [32]:
# ! ls /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Human/

### 5e · Mouse — Gene → BiologicalProcess

In [33]:
# ── Mouse ─────────────────────────────────────────────────────────────────────
Mouse_Gene_BiologicalProcess = DAA_gene_data[DAA_gene_data['Species'] == 'Mus musculus'].copy()
Mouse_Gene_BiologicalProcess['Head_detail_name'] = Mouse_Gene_BiologicalProcess['Head'].map(Mouse_NCBI_Synbol_GENEname_dict)
before = len(Mouse_Gene_BiologicalProcess)
Mouse_Gene_BiologicalProcess = Mouse_Gene_BiologicalProcess[~Mouse_Gene_BiologicalProcess['Head_detail_name'].isna()]
print(f"\nMouse NCBI validation: {len(Mouse_Gene_BiologicalProcess):,} kept / {before - len(Mouse_Gene_BiologicalProcess):,} dropped")

Mouse_Gene_BiologicalProcess['Head_type']       = 'Gene'
Mouse_Gene_BiologicalProcess['Tail_type']       = 'BiologicalProcess'
Mouse_Gene_BiologicalProcess['Head_ID_IS']      = 'NCBI_ID'
Mouse_Gene_BiologicalProcess['Tail_ID_IS']      = 'Quick_GO'
Mouse_Gene_BiologicalProcess['Species']         = 'Mus musculus'
Mouse_Gene_BiologicalProcess['Relation_Source'] = 'DigitalAgingAtlas'

Mouse_Gene_BiologicalProcess = Mouse_Gene_BiologicalProcess[
    [c for c in desired_cols if c in Mouse_Gene_BiologicalProcess.columns]
]

print(f"\nMouse shape: {Mouse_Gene_BiologicalProcess.shape}")
print(Mouse_Gene_BiologicalProcess['Relation'].value_counts())
Mouse_Gene_BiologicalProcess


Mouse NCBI validation: 548 kept / 79 dropped

Mouse shape: (548, 9)
Relation
PositivelyAssociatedWith    307
NegativelyAssociatedWith    217
NotAssociatedWith            24
Name: count, dtype: int64


,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Relation_Source,Species,Data_type
0,Abat,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,4-aminobutyrate aminotransferase,DigitalAgingAtlas,Mus musculus,Associative
5,Dhcr7,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,7-dehydrocholesterol reductase,DigitalAgingAtlas,Mus musculus,Associative
6,Abhd14a,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,abhydrolase domain containing 14A,DigitalAgingAtlas,Mus musculus,Associative
17,Acta1,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,"actin alpha 1, skeletal muscle",DigitalAgingAtlas,Mus musculus,Associative
18,Actc1,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,"actin, alpha, cardiac muscle 1",DigitalAgingAtlas,Mus musculus,Associative
...,...,...,...,...,...,...,...,...,...
4240,Zfp62,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,zinc finger protein 62,DigitalAgingAtlas,Mus musculus,Associative
4241,Zfp64,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,zinc finger protein 64,DigitalAgingAtlas,Mus musculus,Associative
4243,Zfp68,PositivelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,zinc finger protein 68,DigitalAgingAtlas,Mus musculus,Associative
4245,Zfp84,NegativelyAssociatedWith,GO:0007568,Gene,BiologicalProcess,zinc finger protein 84,DigitalAgingAtlas,Mus musculus,Associative


In [34]:
# ── Split and save Mouse ──────────────────────────────────────────────────────

# ── Split and save Human ──────────────────────────────────────────────────────
POS = 'PositivelyAssociatedWith'
NEG = 'NegativelyAssociatedWith'
NOT = 'NotAssociatedWith'

Mouse_Gene_Pos = Mouse_Gene_BiologicalProcess[Mouse_Gene_BiologicalProcess['Relation'] == POS]
Mouse_Gene_Neg = Mouse_Gene_BiologicalProcess[Mouse_Gene_BiologicalProcess['Relation'] == NEG]
Mouse_Gene_Not = Mouse_Gene_BiologicalProcess[Mouse_Gene_BiologicalProcess['Relation'] == NOT]

Mouse_Gene_Pos.to_csv(os.path.join(OUT_MOUSE, 'DAA_Mouse_Gene_Pos_BiologicalProcess.csv'), index=False)
Mouse_Gene_Neg.to_csv(os.path.join(OUT_MOUSE, 'DAA_Mouse_Gene_Neg_BiologicalProcess.csv'), index=False)
Mouse_Gene_Not.to_csv(os.path.join(OUT_MOUSE, 'DAA_Mouse_Gene_Not_BiologicalProcess.csv'), index=False)

print(f"\nSaved 3 Mouse files:")
print(f"  Pos: {len(Mouse_Gene_Pos):,} rows")
print(f"  Neg: {len(Mouse_Gene_Neg):,} rows")
print(f"  Not: {len(Mouse_Gene_Not):,} rows")


Saved 3 Mouse files:
  Pos: 307 rows
  Neg: 217 rows
  Not: 24 rows


In [35]:
OUT_MOUSE

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/digitalagingatlas/Mouse/'

In [36]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/'

---
## 6 · Summary — all output files written

In [37]:
# Walk OUT_PATH and list every CSV produced with row count and file size
print(f"Output files under: {OUT_PATH}\n")
total = 0
for root, dirs, files in os.walk(OUT_PATH):
    dirs.sort()
    for fname in sorted(files):
        if fname.endswith('.csv') and 'DAA_' in fname:
            full = os.path.join(root, fname)
            rows = sum(1 for _ in open(full)) - 1
            size = os.path.getsize(full)
            rel  = os.path.relpath(full, OUT_PATH)
            total += 1
            print(f"  {rel:<70s}  {rows:>6,} rows  {size:>9,} bytes")
print(f"\nTotal: {total} files")

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/

  digitalagingatlas/Human/DAA_Gene_NegativelyAssociatedWith_BiologicalProcess_Human.csv     310 rows     43,938 bytes
  digitalagingatlas/Human/DAA_Gene_NotAssociatedWith_BiologicalProcess_Human.csv       1 rows        228 bytes
  digitalagingatlas/Human/DAA_Gene_PositivelyAssociatedWith_BiologicalProcess_Human.csv     215 rows     29,920 bytes
  digitalagingatlas/Human/DAA_Gene_Tissue_Human.csv                        2,917 rows    375,614 bytes
  digitalagingatlas/Mouse/DAA_Gene_Tissue_Mouse.csv                          576 rows     76,700 bytes
  digitalagingatlas/Mouse/DAA_Mouse_Gene_Neg_BiologicalProcess.csv           217 rows     31,316 bytes
  digitalagingatlas/Mouse/DAA_Mouse_Gene_Not_BiologicalProcess.csv            24 rows      3,450 bytes
  digitalagingatlas/Mouse/DAA_Mouse_Gene_Pos_BiologicalProcess.csv           307 rows     43,556 bytes

Total: 8 files
